In [ ]:
import pandas as pd
import numpy as np

months = ["Січ", "Лют", "Бер", "Кві", "Тра", "Чер", "Лип", "Сер", "Вер", "Жов", "Лис", "Гру"]
avg_temp = [-4.2, -3.1, 1.5, 8.2, 13.8, 16.9, 18.5, 17.8, 12.9, 7.4, 2.1, -2.5]
precip = [44, 42, 46, 58, 90, 105, 100, 82, 62, 52, 56, 50]

df = pd.DataFrame({
    "month": months,
    "avg_temp": avg_temp,
    "precip": precip
})

display(df.head())
print("Shape:", df.shape)
print("\nDtypes:\n", df.dtypes)

In [ ]:
# 1. Виведення DataFrame з індексом за замовчуванням
print("1. df.index за замовчуванням:")
print(df.index)

# 2. Заміна індексу на стовпець month і вивід через .loc рядок для місяця
df_month = df.set_index("month")
print("\n2. Рядок для місяця 'Тра' через .loc:")
display(df_month.loc["Тра"])

# 3. Повернення звичайної нумерації рядків
df_month.reset_index(inplace=True)
print("\n3. Індекс після скидання (reset_index):")
print(df_month.index)

In [ ]:
# Встановлюємо числовий індекс від 1 до 12 через pd.Index (для чистоти Pylance)
df.index = pd.Index(range(1, 13))

# 1. Через .loc виводимо стовпець avg_temp для місяців із підписом від 1 до 6
print("1. Через .loc (підписи від 1 до 6):")
display(df.loc[1:6, "avg_temp"])

# 2. Через .iloc виводимо той самий стовпець за позиціями 0–5
print("\n2. Через .iloc (позиції від 0 до 5):")
display(df.iloc[0:6, df.columns.get_loc("avg_temp")])

# 3. Вивести лише стовпці month і precip (без avg_temp) для всіх рядків
print("\n3. Стовпці month і precip для всіх рядків:")
display(df.loc[:, ["month", "precip"]])

In [ ]:
# 1. Місяці, де avg_temp > 15
print("1. Місяці, де avg_temp > 15:")
display(df[df["avg_temp"] > 15]["month"])

# 2. Місяці, де одночасно avg_temp > 10 і precip > 50 (через &)
print("\n2. Місяці, де avg_temp > 10 та precip > 50:")
display(df[(df["avg_temp"] > 10) & (df["precip"] > 50)]["month"])

# 3. Місяці, де avg_temp < 0 або precip > 80 (через |)
print("\n3. Місяці, де avg_temp < 0 або precip > 80:")
display(df[(df["avg_temp"] < 0) | (df["precip"] > 80)]["month"])

In [ ]:
# 1. Стовпець avg_temp_f — температура у Фаренгейтах векторизованою формулою
df["avg_temp_f"] = df["avg_temp"] * 9/5 + 32

# 2. Булевий стовпець is_wet — True, якщо precip більше середнього значення precip по всій таблиці
mean_precip = df["precip"].mean()
df["is_wet"] = df["precip"] > mean_precip

# 3. Текстовий стовпець season (зима/весна/літо/осінь) за номером місяця через .apply()
def get_season(month_name):
    if month_name in ["Гру", "Січ", "Лют"]:
        return "зима"
    elif month_name in ["Бер", "Кві", "Тра"]:
        return "весна"
    elif month_name in ["Чер", "Лип", "Сер"]:
        return "літо"
    else:
        return "осінь"

df["season"] = df["month"].apply(get_season)

print("Фінальний DataFrame з обчислюваними стовпцями:")
display(df)

### 3. Відповіді на контрольні запитання

* **Чи можете пояснити, чому .loc[1:6] і .iloc[0:6] можуть повернути різну кількість рядків, залежно від того, який саме індекс у таблиці?**  
  Метод `.loc` є включним за міткою (тобто включає рядок із міткою `6`), тоді як `.iloc` працює за класичними позиційними індексами Python (де права межа не включається, тобто бере позиції від 0 до 5). Крім того, якщо мітки індексу відрізняються від звичайного порядку, `.loc` шукатиме точні збіги міток.

* **Чи розумієте, чому df["avg_temp"] > 10 and df["precip"] > 50 викличе помилку, а (df["avg_temp"] > 10) & (df["precip"] > 50) — ні?**  
  Оператор `and` у Python оцінює істинність усього об'єкта Series як єдиного цілого булевого значення, що викликає `ValueError` для масивів. Побітовий оператор `&` виконує поелементне порівняння кожного рядка окремо всередині DataFrame.

* **Чи можете описати словами, у чому різниця між обчислюваним стовпецем, створеним формулою, і створеним через .apply()?**  
  Векторизована формула обробляється внутрішніми механізмами бібліотеки на рівні C-коду одразу для всього стовпця (дуже швидко). Метод `.apply()` застосовує передану функцію поелементно до кожного рядка, що зручно для складної логіки з розгалуженнями, але працює повільніше.